In [21]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
import torch

In [22]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [23]:
len(words)

32033

In [24]:
# build the vocabulary of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

itos = {i:s for s, i in stoi.items()}

In [92]:
#build the dataset

block_size = 3 #Context- How many Characters do we need to predict the next one?
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->' , itos[ix])
        context = context[1:] + [ix]
        
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


torch.Size([32])

In [29]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [56]:
C = torch.randn((27, 2))

In [77]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [78]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [79]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h

tensor([[ 0.9992,  0.8007,  0.5714,  ..., -0.3269,  0.9945, -0.1181],
        [ 0.9951,  1.0000,  0.9998,  ...,  0.8701, -0.3197, -0.9526],
        [ 0.9678, -0.3651,  0.9372,  ..., -0.9945, -0.9809,  0.8951],
        ...,
        [ 0.9721, -0.9997, -0.9951,  ..., -1.0000,  0.9761,  0.4862],
        [ 0.9989,  1.0000,  0.9773,  ...,  1.0000,  0.9962, -0.7152],
        [ 0.9979,  0.2570,  0.6221,  ..., -0.9995,  0.4502, -0.7745]])

In [80]:
W2 = torch.randn(100,27)
b2 = torch.randn(27)

In [81]:
logits = h @ W2 + b2

In [82]:
logits.shape

torch.Size([32, 27])

In [83]:
counts = logits.exp()

In [86]:
prob = counts/counts.sum(1, keepdims=True)

tensor(1.)

In [91]:
prob.shape

torch.Size([32, 27])

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean()